In this work, we use transformer model to integrate gene expression and TCR amino acid sequences

Getting gene data

In [1]:
# %matplotlib inline

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np

import pandas as pd
# import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc

import anndata as ad

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

sc.settings.verbosity = 3


In [2]:
gene_TCR = ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides.h5ad')
gene_TCR

/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [3]:
gene = pd.DataFrame(gene_TCR.layers['raw_counts'].todense())
gene

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
4,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
145475,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,5.0,3.0,0.0,0.0
145476,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
145477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0


In [4]:
tcr_seq = gene_TCR.obs[['cdr3_TRB']]
tcr_seq

,cdr3_TRB
barcode,
AGGGTGAGTATTACCG-18,CSAPSGEGRDTQYF
CTTGGCTTCGTTGCCT-25,CASSLFDSQETQYF
ACGATACTCGCAGGCT-40,CASSLFDSGRLDTQYF
ACGCCAGTCATGTCTT-8,CSASPGDYEQYF
TTCTTAGCAAAGAATC-4,CASSHGKGGNEQFF
...,...
GAAGCAGAGCAGGCTA-3,CATSDRLAGGELFF
CAGTCCTTCATCACCC-8,CASSYLAGDFTDTQYF
GACTACACACGGTAAG-3,CASRTGLASTDTQYF


In [5]:
import tensorflow as tf

2026-03-03 15:43:55.272667: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 15:43:55.424094: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 15:43:55.455882: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [16]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
from tensorflow.keras.initializers import HeNormal
# Define input layer
input_gex = Input(shape=(100,))
gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
gex = Reshape(target_shape=(8,8,1))(gex)

# Convolutional layers
gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

gex = Flatten()(gex)
hidden_layer = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(gex)

# Transposed Convolutional layers
tcr = Reshape(target_shape=(8,8,1))(hidden_layer)
tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Dropout(rate=0.2)(tcr)
tcr = Flatten()(tcr)
tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
tcr = Dense(units=121, activation='relu')(tcr)
# Define model
model = Model(inputs=input_gex, outputs=tcr)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00015),
              loss='mse')

# Check layer names
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 100)]             0         
                                                                 
 dense_10 (Dense)            (None, 64)                6464      
                                                                 
 reshape_4 (Reshape)         (None, 8, 8, 1)           0         
                                                                 
 conv2d_4 (Conv2D)           (None, 8, 8, 64)          640       
                                                                 
 conv2d_5 (Conv2D)           (None, 8, 8, 32)          18464     
                                                                 
 flatten_4 (Flatten)         (None, 2048)              0         
                                                                 
 dense_11 (Dense)            (None, 64)                1311

In [7]:
# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten
# from tensorflow.keras.initializers import HeNormal

# # Define input layer
# input_gex = Input(shape=(100,))
# gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
# gex = Reshape(target_shape=(8,8,1))(gex)

# # Convolutional layers
# gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
# gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

# gex = Flatten()(gex)
# hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# # Transposed Convolutional layers
# tcr = Reshape(target_shape=(8,8,1))(hidden_layer)
# tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
# tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)

# tcr = Flatten()(tcr)
# tcr = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(tcr)
# tcr = Dense(units=121)(tcr)  # Ensure proper activation

# # Define model
# model = Model(inputs=input_gex, outputs=tcr)
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#               loss='mse')

# # Check layer names
# model.summary()


In [8]:
AE_tcr = pd.read_csv("../AE_emb_TRB_all_peptides_10X_all_donors.csv")
AE_tcr

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,-1541.692100,431.963380,-113.48751,363.83960,-617.465500,-576.38770,-152.861240,-985.72640,445.512240,-236.100420,...,28.279720,1018.37134,261.223200,87.342865,-228.999980,146.428400,532.794070,865.979250,-578.558960,855.616900
1,-273.513100,-398.893860,616.80430,-877.82350,-871.226700,375.48306,422.579040,192.50789,-199.132490,-265.096560,...,494.893800,258.40955,-404.777220,-428.136350,-191.204730,-372.523960,203.390100,1186.992800,96.218480,-457.060200
2,380.871000,58.110577,-132.31460,-259.36774,-274.578430,-194.46793,248.315840,265.90674,-60.173637,-98.568726,...,92.126270,29.88193,-891.704800,370.281830,59.372814,-729.678000,902.504940,1161.101400,-700.613340,-425.839780
3,-1229.993200,261.055080,-116.35792,339.64825,-440.477600,65.39269,-221.381640,-939.52580,145.774370,39.421880,...,153.995830,770.41030,-417.146600,194.590100,-287.056100,-4.876206,-74.260704,43.650208,-67.906160,-309.779940
4,-1113.337300,-93.408485,-280.13140,-150.60530,-93.476110,-520.02313,64.995720,-1008.05770,174.263440,-62.470676,...,-56.364480,984.19050,467.822940,247.904310,409.301880,-60.920048,744.377300,1218.065600,-456.735320,617.486100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,-512.594500,-576.577700,-841.85020,-472.11673,-47.690334,-245.14507,299.702820,-409.62906,-274.262500,191.512990,...,-605.121150,1250.64900,-388.322940,282.790900,-704.643740,37.460022,647.884160,1549.309400,352.393280,-108.645310
145475,226.274380,956.163450,176.05064,-689.92920,162.394490,608.82790,-269.477480,-744.45300,-219.842730,835.156560,...,31.987225,508.31207,-341.937840,-267.425480,-786.679300,-816.931300,291.385250,466.537630,-74.314735,-342.459630
145476,1080.433300,373.862700,-65.57249,-259.77700,35.480648,-554.94806,-441.794400,93.13507,-163.760620,155.478730,...,-251.648400,687.42706,-52.241924,-690.953600,523.990230,-314.377440,-316.330050,697.732100,40.809692,-1766.797400
145477,55.882103,-766.636600,229.29408,-177.78737,634.726900,-295.03433,92.827480,443.62762,244.667330,-787.343600,...,385.697330,895.55630,-700.939200,-126.450980,-803.600100,133.600450,-130.163740,348.644740,-400.828250,38.264355


In [9]:
# import numpy as np
# from sklearn.decomposition import NMF

# # Generate random non-negative data
# data = gene.to_numpy()

# # Initialize the NMF model
# n_components = 100
# model_nmf = NMF(n_components=n_components, init='random', random_state=0)

# # Fit the model to the data
# W = model_nmf.fit_transform(data)
# H = model_nmf.components_

# # Display the results
# print("Basis matrix (W):\n", W)
# print("Coefficients matrix (H):\n", H)


In [10]:

# W = pd.DataFrame(W)
# W

In [11]:
# W.to_csv('gex_nmf_100_components_all_peptides_10X_all_donors.csv', index=False)
W = pd.read_csv('gex_nmf_100_components_all_peptides_10X_all_donors.csv')
W

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.227333,7.086625,0.466240,0.436161,0.334158,0.003627,0.198576,0.001280,0.004045,0.002781,...,0.004944,0.033126,0.143294,0.116318,0.053315,0.090684,0.018362,0.296472,0.013140,0.083483
1,1.316014,5.126111,0.649829,0.694511,0.023224,0.000310,0.057097,0.012929,0.008026,0.030475,...,0.007351,0.088509,0.556798,0.192283,0.042265,0.103002,0.001494,0.392443,0.000000,0.052590
2,0.000000,0.000000,0.405181,0.836287,4.124795,0.007945,0.000000,0.003714,0.002770,0.026438,...,0.006715,0.044787,0.544515,0.088714,0.143255,0.056269,0.018553,0.658247,0.050502,0.113769
3,0.000000,2.308417,0.119431,0.120097,3.985985,0.015477,0.000000,0.004406,0.025324,0.015539,...,0.009446,0.125159,0.000000,0.071700,0.156441,0.074255,0.014709,0.105227,0.006118,0.081632
4,0.000000,0.000000,0.546531,1.292965,3.482252,0.010140,0.046020,0.018085,0.022204,0.046295,...,0.061491,0.027266,0.298706,0.421875,0.362350,0.100456,0.010704,0.897931,0.000000,0.181201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,0.000000,0.000000,0.132730,0.000000,0.000000,0.000000,0.024647,0.000000,0.000000,0.001994,...,0.002062,0.000000,0.025164,0.052784,0.000000,0.000000,0.000000,0.171280,0.000000,0.066642
145475,0.000000,2.239055,0.186252,0.369426,0.800173,0.016219,0.026338,0.019682,0.022974,0.000450,...,0.002169,0.031499,0.040721,0.270443,0.118005,0.139055,0.019971,0.630590,0.052803,0.172927
145476,0.024406,0.937475,0.406066,0.069124,2.025419,0.009820,0.002730,0.004958,0.010678,0.029238,...,0.000000,0.024329,0.136838,0.133530,0.076204,0.023668,0.002195,0.451703,0.004854,0.133263
145477,0.000000,0.000000,0.244838,0.000000,0.000000,0.002488,0.039718,0.000000,0.001914,0.000000,...,0.000000,0.004401,0.046355,0.000000,0.011181,0.003384,0.009216,0.004487,0.000000,0.025856


In [17]:

# es_callback = EarlyStopping(monitor= 'val_auc', patience=20, restore_best_weights=True)
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=25, monitor='loss', min_delta=100)

history = model.fit(W,AE_tcr, 
                epochs=3000, 
                batch_size=256, 
                shuffle = True,
                # callbacks=[es_callback, checkpoint,reduce_learning_rate])
                # callbacks=[reduce_learning_rate]
                )

Epoch 1/3000
569/569 [==============================] - 3s 4ms/step - loss: 265401.8750
Epoch 2/3000
569/569 [==============================] - 2s 4ms/step - loss: 247646.0938
Epoch 3/3000
569/569 [==============================] - 2s 4ms/step - loss: 247610.5000
Epoch 4/3000
569/569 [==============================] - 2s 4ms/step - loss: 247602.3906
Epoch 5/3000
569/569 [==============================] - 2s 4ms/step - loss: 247586.0938
Epoch 6/3000
569/569 [==============================] - 2s 4ms/step - loss: 247579.6250
Epoch 7/3000
569/569 [==============================] - 2s 4ms/step - loss: 247575.0000
Epoch 8/3000
569/569 [==============================] - 2s 4ms/step - loss: 247564.3281
Epoch 9/3000
569/569 [==============================] - 2s 4ms/step - loss: 247536.6875
Epoch 10/3000
569/569 [==============================] - 2s 4ms/step - loss: 247505.8594
Epoch 11/3000
569/569 [==============================] - 2s 4ms/step - loss: 247478.4219
Epoch 12/3000
569/569 [=======

In [18]:
for layer in model.layers:
    print(layer.name)

input_3
dense_10
reshape_4
conv2d_4
conv2d_5
flatten_4
dense_11
reshape_5
conv2d_transpose_4
conv2d_transpose_5
dropout_2
flatten_5
dense_12
dense_13
dense_14


In [19]:
from tensorflow.keras.models import Model
latent_model = Model(inputs=input_gex, outputs=hidden_layer)


In [20]:
model.predict( W.iloc[1:2])

1/1 [==============================] - 0s 281ms/step


array([[1.4161285e+00, 1.4728477e+01, 0.0000000e+00, 0.0000000e+00,
        1.1333625e+02, 1.3411736e+02, 0.0000000e+00, 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 2.2403737e+02,
        3.4631618e+01, 1.0363980e+01, 0.0000000e+00, 8.1015961e+01,
        0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 1.7631659e+02,
        0.0000000e+00, 6.7272713e+01, 0.0000000e+00, 1.3567583e+01,
        0.0000000e+00, 0.0000000e+00, 2.1936079e+02, 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 2.9494916e+02,
        9.7496510e-03, 4.1074982e+00, 2.1382764e+02, 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00, 1.7310286e+01, 2.2469633e+01,
        3.3231320e+02, 0.0000000e+00, 4.9780823e+01, 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00, 1.2952959e+02, 5.0697479e+01,
        5.8972458e+01, 2.3102707e+02, 0.0000000e+00, 7.0421692e+01,
        1.8198050e+02, 0.0000000e+00, 0.0000000e+00, 1.0037173e+02,
        0.0000000e+00, 6.3402172e+01, 0.0000000e

In [21]:
model.predict( W.iloc[4:5])

1/1 [==============================] - 0s 29ms/step


array([[   0.      ,  174.24878 ,    0.      ,    0.      ,    0.      ,
           0.      ,    0.      ,    0.      ,    0.      ,   21.098475,
           0.      ,  473.93912 ,    0.      ,  310.0596  ,   19.797068,
          19.892818,    0.      ,   33.72309 ,    0.      ,  451.73447 ,
          79.50054 ,    0.      ,    0.      ,    0.      ,    0.      ,
           0.      ,  567.37665 ,    0.      ,    0.      ,    0.      ,
           0.      ,  488.4411  ,    0.      ,  406.46127 ,  226.22427 ,
           0.      ,  147.11958 ,    0.      ,  119.98261 ,  219.60951 ,
         566.93164 ,    0.      ,   77.6459  ,    0.      ,  150.1516  ,
           0.      ,  344.30066 ,    0.      ,  127.53725 ,  455.30582 ,
          80.05705 ,  161.16725 ,  416.7667  ,    0.      ,    0.      ,
         108.2167  ,   30.950115,  328.0904  ,   91.58552 ,  153.59523 ,
           0.      ,  260.20523 ,    0.      ,    0.      ,   24.666965,
           0.      ,    0.      , 3600.0847  ,    0

In [22]:
integration_pred = latent_model.predict( W)

4547/4547 [==============================] - 5s 983us/step


In [23]:
pd.DataFrame(integration_pred[1:50,1:50])

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,0.000000,0.0,0.0,3.701933,9.362439,13.073524,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,1.064706,0.0,0.000000,0.000000,3.154169
1,0.000000,0.0,0.0,1.647828,2.164470,14.929717,0.0,0.0,0.000000,0.0,...,3.197256,0.000000,0.000000,0.0,0.000000,5.914854,0.0,3.724772,0.000000,0.000000
2,0.942395,0.0,0.0,4.074978,7.017201,4.498881,0.0,0.0,3.047668,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,0.718090,0.0,0.000000,0.000000,0.000000
3,6.829710,0.0,0.0,0.000000,0.033535,4.822514,0.0,0.0,2.093441,0.0,...,1.865396,0.000000,16.174997,0.0,0.000000,4.308629,0.0,5.155125,2.983720,0.000000
4,2.266372,0.0,0.0,1.287762,3.185766,4.524041,0.0,0.0,0.093549,0.0,...,2.064587,0.000000,3.668772,0.0,0.000000,5.566699,0.0,1.957512,5.690331,0.000000
5,0.000000,0.0,0.0,0.085645,0.547467,4.704464,0.0,0.0,0.000000,0.0,...,1.703755,0.000000,0.000000,0.0,0.381489,2.377525,0.0,5.272830,0.000000,2.012871
6,0.000000,0.0,0.0,0.000000,6.943415,8.848470,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,5.178318,0.0,0.547465,0.698390,0.000000
7,0.176724,0.0,0.0,1.368825,3.738276,0.368692,0.0,0.0,1.994269,0.0,...,0.000000,0.000000,0.000000,0.0,3.943576,4.909998,0.0,0.000000,2.257046,1.336611
8,0.000000,0.0,0.0,0.000000,0.772625,1.925091,0.0,0.0,0.000000,0.0,...,0.000000,1.209625,3.322177,0.0,0.633170,4.440424,0.0,3.146820,2.106181,0.000000
9,0.000000,0.0,0.0,2.363780,5.307608,8.065827,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,0.0,0.000000,4.300633,0.0,2.084829,0.000000,0.000000


In [24]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])

In [25]:
integration_pred.shape

(145479, 64)

In [26]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])
integration_pred.shape

(145479, 64)

In [27]:
pd.DataFrame(integration_pred)

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,4.672718,0.000000,0.0,0.0,6.857419,1.872721,6.280306,0.0,0.0,2.004314,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000
1,0.000000,0.000000,0.0,0.0,3.701933,9.362439,13.073524,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,10.413274
2,2.364559,0.000000,0.0,0.0,1.647828,2.164470,14.929717,0.0,0.0,0.000000,...,0.0,3.946451,0.0,0.0,0.000000,0.0,1.942125,0.000000,0.0,6.559614
3,9.046071,0.942395,0.0,0.0,4.074978,7.017201,4.498881,0.0,0.0,3.047668,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.118451,0.0,2.342225
4,0.000000,6.829710,0.0,0.0,0.000000,0.033535,4.822514,0.0,0.0,2.093441,...,0.0,0.000000,0.0,0.0,12.873353,0.0,3.674380,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,0.000000,1.919112,0.0,0.0,2.697479,8.671455,5.111042,0.0,0.0,0.296422,...,0.0,4.186335,0.0,0.0,0.000000,0.0,1.781378,6.192757,0.0,0.806127
145475,0.000000,0.000000,0.0,0.0,2.046324,5.926474,10.747284,0.0,0.0,0.000000,...,0.0,0.821332,0.0,0.0,0.000000,0.0,5.394743,1.891414,0.0,10.415553
145476,0.000000,2.299400,0.0,0.0,0.000000,1.161693,1.747009,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,7.276559,2.727138,0.0,8.348395
145477,0.000000,0.000000,0.0,0.0,0.000000,7.797906,8.841404,0.0,0.0,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.0,2.543676,0.305229,0.0,4.229376


In [28]:
pd.DataFrame(integration_pred).to_csv("integration_pred_new_method_gex_to_TCR_beta_chain_10X_all_peptides_64_03032026.csv", index=False)